# Capítulo 7: Trabalhando com Dados

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 10 de Grus (2019).

> Especialistas costumam ter mais dados do que juízo.
>
> — Colin Powell

O [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) construiu a máquina que ajusta parâmetros. O [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) construiu os canos pelos quais o dado chega até um programa Python. Nenhum dos dois te prepara para o que acontece entre uma coisa e outra: o dado que chegou pelos canos do Capítulo 6 quase nunca está pronto para alimentar a máquina do Capítulo 5. Ele vem torto, incompleto, em unidades incompatíveis, com colunas demais ou de menos. Este capítulo é sobre essa distância.

É um capítulo mais de artesanato do que de teoria. Não há um modelo novo para treinar nem uma prova para seguir — em vez disso, um punhado de técnicas pequenas e recorrentes: como olhar para um conjunto de dados antes de fazer qualquer coisa com ele, como representar uma linha de dados heterogênea sem reescrever a mesma lógica de conversão de tipo em cada função, como filtrar o que está sujo sem descartar o que só *parece* sujo, como agregar e comparar, como colocar dimensões em pé de igualdade antes de medir distância entre elas — e, por fim, como reduzir um conjunto de muitas dimensões a poucas, sem perder o que importa.

Ao final deste capítulo, você será capaz de:

- Resumir e visualizar um conjunto de dados de uma, duas e muitas dimensões antes de tentar modelá-lo
- Representar uma linha de dados heterogênea com `NamedTuple`, e explicar por que isso resolve os dois problemas que um `dict` tem para essa tarefa
- Escrever o equivalente mutável com `dataclass`, e reconhecer o compromisso que a mutabilidade reintroduz
- Escrever uma função de *parsing* que devolve `None` diante de dado ruim em vez de estourar, e decidir o que fazer com as linhas rejeitadas
- Agrupar, ordenar e agregar dados tabulares com `defaultdict` e compreensões, no mesmo estilo do [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/index.html)
- Reescalonar dados para que a unidade de cada dimensão pare de dominar o cálculo de distância
- Reduzir a dimensionalidade de um conjunto de dados com PCA, construído do zero sobre o gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html)

## Seções

| Seção | Tópico |
|---|---|
| [7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) | Explorando Seus Dados |
| [7.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/02-namedtuples.html) | Usando NamedTuples |
| [7.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-dataclasses.html) | Dataclasses |
| [7.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-limpeza-e-transformacao.html) | Limpeza e Transformação |
| [7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-manipulando-dados.html) | Manipulando Dados |
| [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) | Reescalonamento |
| [7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-um-parenteses-tqdm.html) | Um Parêntese: tqdm |
| [7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) | Redução de Dimensionalidade |

A última seção também cumpre uma promessa: o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html), ao discutir seleção de atributos, cita a redução de dimensionalidade "no Capítulo 7" como uma das formas de lidar com dados que têm atributos demais. É esta seção.

## Explorando Seus Dados

> **📌 Nota**
>
> Esta seção corresponde a *Exploring Your Data*, do capítulo 10 de Grus (2019).

Depois de identificar as perguntas que você quer responder e conseguir dado nas mãos — o assunto do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) —, a tentação é pular direto para os modelos. Resista. O primeiro passo deveria ser sempre **explorar** os dados.

### Dados unidimensionais

O caso mais simples é um conjunto de dados unidimensional: só uma coleção de números — o tempo médio diário que cada usuário passa no seu site, por exemplo. Um primeiro passo óbvio é calcular estatísticas resumo: quantos pontos você tem, o menor, o maior, a média, o desvio padrão.

Mas mesmo essas estatísticas não necessariamente dão uma boa noção do que está acontecendo. Um passo melhor é construir um **histograma**, agrupando os dados em faixas discretas — *buckets* — e contando quantos pontos caem em cada uma:

In [ ]:
from typing import List, Dict
from collections import Counter
import math

def bucketize(point: float, bucket_size: float) -> float:
    """Arredonda o ponto para baixo até o múltiplo mais próximo de bucket_size"""
    return bucket_size * math.floor(point / bucket_size)

def make_histogram(points: List[float], bucket_size: float) -> Dict[float, int]:
    """Distribui os pontos nos buckets e conta quantos caem em cada um"""
    return Counter(bucketize(point, bucket_size) for point in points)

> **🟩 Exemplo**
>
> Repare que `make_histogram` é a mesma ideia do histograma de notas do [Capítulo 3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap03/02-graficos-de-barras.html): um `Counter` sobre valores arredondados para o piso do bucket. A única diferença é que aqui o arredondamento vira uma função nomeada, `bucketize`, em vez de uma expressão solta como `grade // 10 * 10` — o que compensa quando o tamanho do bucket muda de um gráfico para o outro, como acontece logo abaixo.

Para desenhar, basta um gráfico de barras — o mesmo `plt.bar` do Capítulo 3:

In [ ]:
from matplotlib import pyplot as plt

def plot_histogram(points: List[float], bucket_size: float, title: str = ""):
    histogram = make_histogram(points, bucket_size)
    plt.bar(histogram.keys(), histogram.values(), width=bucket_size)
    plt.xlabel("valor")
    plt.ylabel("frequência")
    plt.title(title)

Considere dois conjuntos de dados fictícios: um uniforme entre -100 e 100, e um normal com média 0 e desvio padrão 57. Para o segundo, precisamos da inversa da normal acumulada — dada uma probabilidade, ela devolve o valor abaixo do qual a normal padrão cai com aquela probabilidade. Chama-se `inverse_normal_cdf` e vem de `scratch/probability.py`.

In [ ]:
from scratch.probability import inverse_normal_cdf
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
import random

def random_normal() -> float:
    """Uma amostra da normal padrão"""
    return inverse_normal_cdf(random.random())

random.seed(0)

# uniforme entre -100 e 100
uniform = [200 * random.random() - 100 for _ in range(10000)]

# normal com média 0, desvio padrão 57
normal = [57 * random_normal() for _ in range(10000)]

f"médias: {sum(uniform)/len(uniform):.2f} e {sum(normal)/len(normal):.2f}"

Ambos têm média perto de 0. Os dois histogramas mostram o quanto isso esconde:

In [ ]:
# Figura: Histograma da distribuição uniforme
plot_histogram(uniform, 10, "Histograma — uniforme")
plt.show()

In [ ]:
# Figura: Histograma da distribuição normal
plot_histogram(normal, 10, "Histograma — normal")
plt.show()

Os dois também têm desvio padrão próximo de 57. Ainda assim, um é achatado e limitado; o outro tem pico ao centro e caudas compridas. Nenhuma estatística resumo diz isso — só olhar para a forma diz.

### Duas dimensões

Com duas dimensões, você quer entender cada uma individualmente, mas também como elas se relacionam. Considere mais um conjunto fictício:

In [ ]:
random.seed(4)
xs = [random_normal() for _ in range(1000)]
ys1 = [ x + random_normal() / 2 for x in xs]
ys2 = [-x + random_normal() / 2 for x in xs]

Se você rodasse `plot_histogram` em `ys1` e `ys2` separadamente, teria dois histogramas parecidos — de fato, os dois têm distribuição normal, com a mesma média e o mesmo desvio padrão. Mas cada um tem uma relação muito diferente com `xs`, e isso só aparece quando você olha as duas dimensões juntas:

In [ ]:
# Figura: Duas distribuições conjuntas muito diferentes
plt.scatter(xs, ys1, marker='.', color='black', label='ys1')
plt.scatter(xs, ys2, marker='.', color='gray',  label='ys2')
plt.xlabel('xs')
plt.ylabel('ys')
plt.legend(loc=9)
plt.title("Distribuições conjuntas muito diferentes")
plt.show()

`ys1` cresce com `xs`; `ys2` decresce. A diferença fica evidente também na correlação:

In [ ]:
from scratch.statistics import correlation
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
correlation(xs, ys1), correlation(xs, ys2)

> **⚠️ Atenção — Uma afirmação sobre dado aleatório precisa de semente**
>
> Suponha que você quisesse registrar esse resultado como um teste, do jeito que este livro faz o tempo todo:
>
> ```python
> assert 0.89 < correlation(xs, ys1) < 0.91
> assert -0.91 < correlation(xs, ys2) < -0.89
> ```
>
> Se `xs`, `ys1` e `ys2` fossem gerados **sem** fixar semente, esse teste seria cara ou coroa. O valor de `correlation(xs, ys1)` para essa construção gira em torno de 0,894 — perto o bastante da borda inferior da janela `(0.89, 0.91)` para que o `assert` falhe cerca de uma vez em cada quatro execuções (medido: 24,6% e 25,0% em duas séries independentes de 20 mil repetições). Não seria um defeito no cálculo da correlação: é uma afirmação sobre uma amostra aleatória, escrita como se fosse determinística.
>
> São duas lições, e valem para qualquer código seu: uma afirmação sobre dado aleatório só é reprodutível se a semente estiver fixada, e uma janela de tolerância só é segura se for larga o bastante para a variação real da amostra, não só para o valor obtido numa execução específica. Por isso `random.seed(4)` aparece aqui em cima, antes de `xs`: com a semente fixada, a correlação sai sempre a mesma — cerca de 0,906 e -0,907 — em qualquer execução.

### Muitas dimensões

Com muitas dimensões, você quer saber como todas elas se relacionam entre si. Uma abordagem simples é a **matriz de correlação**, cuja entrada na linha $i$, coluna $j$ é a correlação entre a dimensão $i$ e a dimensão $j$ dos dados:

In [ ]:
from scratch.linear_algebra import Matrix, Vector, make_matrix

def correlation_matrix(data: List[Vector]) -> Matrix:
    """
    Devolve a matriz len(data) x len(data) cuja entrada (i, j)
    é a correlação entre data[i] e data[j]
    """
    def correlation_ij(i: int, j: int) -> float:
        return correlation(data[i], data[j])

    return make_matrix(len(data), len(data), correlation_ij)

Uma abordagem mais visual — quando o número de dimensões não é grande demais — é a **matriz de dispersão**, mostrando todos os gráficos de dispersão dois a dois. Para gerar um exemplo com correlações interessantes, construímos quatro séries relacionadas entre si de propósito:

In [ ]:
# Figura: Matriz de dispersão de quatro séries correlacionadas
def random_row() -> List[float]:
    row = [0.0, 0, 0, 0]
    row[0] = random_normal()
    row[1] = -5 * row[0] + random_normal()
    row[2] = row[0] + row[1] + 5 * random_normal()
    row[3] = 6 if row[2] > -2 else 0
    return row

random.seed(0)
num_points = 100
corr_rows = [random_row() for _ in range(num_points)]

# cada linha tem 4 pontos, mas queremos as colunas
corr_data = [list(col) for col in zip(*corr_rows)]

num_vectors = len(corr_data)
fig, ax = plt.subplots(num_vectors, num_vectors, figsize=(8, 8))

for i in range(num_vectors):
    for j in range(num_vectors):
        if i != j:
            ax[i][j].scatter(corr_data[j], corr_data[i], s=8)
        else:
            ax[i][j].annotate("série " + str(i), (0.5, 0.5),
                              xycoords='axes fraction',
                              ha="center", va="center")
        if i < num_vectors - 1: ax[i][j].xaxis.set_visible(False)
        if j > 0: ax[i][j].yaxis.set_visible(False)

ax[-1][-1].set_xlim(ax[0][-1].get_xlim())
ax[0][0].set_ylim(ax[0][1].get_ylim())
plt.show()

In [ ]:
matriz = correlation_matrix(corr_data)
for linha in matriz:
    print([round(v, 2) for v in linha])

A matriz confirma o que os gráficos sugerem: a série 1 é fortemente **anticorrelacionada** com a série 0 (a própria fórmula de `row[1]` inverte o sinal de `row[0]`); a série 2 é positivamente correlacionada com a série 1; e a série 3 só assume os valores 0 e 6 — 6 quando a série 2 é grande, 0 quando é pequena —, o que explica as duas faixas de pontos nos gráficos que a envolvem: verticais quando a série 3 está no eixo $x$, horizontais quando está no eixo $y$.

> **🔷 Conceito**
>
> A matriz de dispersão é uma ferramenta de **triagem**, não de conclusão. Ela mostra rápido quais pares de dimensões merecem uma olhada mais de perto — o que é valioso justamente porque examinar cada par manualmente não escala além de um punhado de dimensões. A [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) lida com o problema oposto: dimensões demais para sequer desenhar uma matriz dessas.

> **💡 Dica — Na prática: `pandas` e `seaborn`**
>
> Você acabou de escrever, à mão, o que `pandas.DataFrame.describe()`, `.hist()` e `.corr()` fazem em uma linha cada — e o que `pd.plotting.scatter_matrix(df)` ou `seaborn.pairplot(df)` fazem no lugar da dupla de laços `for i in range(num_vectors): for j in range(num_vectors):` acima.
>
> O que essas funções escondem, e vale saber que está sendo escondido: `.corr()` por padrão calcula a correlação de Pearson — a mesma que você implementou —, mas aceita trocar para Spearman ou Kendall com um parâmetro, que medem relação *monotônica* em vez de linear, e lidam melhor com outliers. `.describe()` decide sozinho quais estatísticas mostrar dependendo do tipo da coluna. E tanto `.hist()` quanto `scatter_matrix` decidem o número de buckets ou o tamanho dos pontos por uma heurística — a mesma heurística que, se você tivesse implementado `bucketize` com um `bucket_size` ruim, teria escondido de você exatamente o tipo de diferença que a seção "Dados unidimensionais" acima existe para revelar.

## Usando NamedTuples

> **📌 Nota**
>
> Esta seção corresponde a *Using NamedTuples*, do capítulo 10 de Grus (2019).

Uma forma comum de representar um registro de dados é um `dict`:

In [ ]:
import datetime

stock_price = {'closing_price': 102.06,
               'date': datetime.date(2014, 8, 29),
               'symbol': 'AAPL'}

Existem boas razões para isso ser menos ideal do que parece.

A primeira é desempenho: um `dict` carrega overhead que uma estrutura mais enxuta evitaria — mas, na maioria dos casos, isso é secundário perto do problema seguinte.

A segunda razão é mais séria: acessar campos por chave de `dict` é propenso a erro. O código abaixo roda **sem** erro nenhum, e faz a coisa errada silenciosamente:

In [ ]:
# opa, erro de digitação
stock_price['cosing_price'] = 103.06
'closing_price' in stock_price, 'cosing_price' in stock_price

`stock_price` agora tem duas chaves parecidas, `closing_price` e `cosing_price`, e nada avisou. Qualquer código que dependa de `stock_price['closing_price']` continua lendo o valor antigo (102.06), enquanto o valor novo (103.06) fica escondido atrás de uma chave com erro de digitação.

Por fim, embora seja possível anotar o tipo de um `dict` uniforme —

In [ ]:
from typing import Dict
prices: Dict[datetime.date, float] = {}

— não existe um jeito útil de anotar um `dict` que representa dado heterogêneo, como o `stock_price` acima, que mistura `str`, `datetime.date` e `float` num único registro. A anotação de tipo perde exatamente o poder que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) apresentou: dizer, antes de ler o corpo do código, o que cada campo é.

### `namedtuple`

Python tem uma classe embutida para isso: `namedtuple`, que é como uma tupla, mas com posições nomeadas.

In [ ]:
from collections import namedtuple

StockPrice = namedtuple('StockPrice', ['symbol', 'date', 'closing_price'])
price = StockPrice('MSFT', datetime.date(2018, 12, 14), 106.03)

assert price.symbol == 'MSFT'
assert price.closing_price == 106.03

Como tuplas comuns, `namedtuple`s são **imutáveis** — não dá para modificar um valor depois de criado. Isso às vezes atrapalha, mas na maior parte das vezes é uma vantagem: um registro que não muda por baixo dos seus pés é mais fácil de raciocinar sobre.

O que `namedtuple` ainda não resolve é a anotação de tipo — `StockPrice` acima não diz, em lugar nenhum, que `closing_price` é `float`. É para isso que existe a variante tipada, `NamedTuple`, do módulo `typing`:

In [ ]:
from typing import NamedTuple

class StockPrice(NamedTuple):
    symbol: str
    date: datetime.date
    closing_price: float

    def is_high_tech(self) -> bool:
        """É uma classe — então também dá para acrescentar métodos"""
        return self.symbol in ['MSFT', 'GOOG', 'FB', 'AMZN', 'AAPL']

price = StockPrice('MSFT', datetime.date(2018, 12, 14), 106.03)

assert price.symbol == 'MSFT'
assert price.closing_price == 106.03
assert price.is_high_tech()

Agora `price.clo` faria seu editor sugerir `closing_price` — porque o editor sabe, pela anotação, que esse campo existe e o que ele é. Um `dict` nunca dá essa pista: `stock_price['clo` não tem como o editor completar, porque chaves de `dict` não são parte da assinatura do tipo.

> **❗ Importante**
>
> `price.cosing_price = 103.06` — o mesmo erro de digitação de antes — agora levanta `AttributeError` na hora, em vez de criar silenciosamente um segundo campo. É a mesma classe de proteção que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) atribuiu às anotações de tipo em geral: elas não impedem todo erro, mas tornam alguns deles impossíveis de passar despercebidos.

> **🟩 Exemplo**
>
> Você vai ver este exato padrão de novo no [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html), sobre k-vizinhos mais próximos:
>
> ```python
> class LabeledPoint(NamedTuple):
>     point: Vector
>     label: str
> ```
>
> `LabeledPoint` empacota um ponto e o rótulo dele exatamente como `StockPrice` empacota um símbolo, uma data e um preço — um registro pequeno, imutável, com nomes de campo que o editor entende. É o uso mais comum de `NamedTuple` neste livro: dar nome e tipo a algo que, de outra forma, seria uma tupla anônima de posições que só quem escreveu a função lembra o que significam.

> **💡 Dica — Na prática: o que se faz com isso**
>
> `NamedTuple` resolve o problema do registro heterogêneo tipado, mas continua imutável — o que é uma escolha, não uma limitação a se contornar. Mutabilidade, sem abrir mão da anotação de tipo, é o assunto da [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-dataclasses.html): `dataclass`.

## Dataclasses

> **📌 Nota**
>
> Esta seção corresponde a *Dataclasses*, do capítulo 10 de Grus (2019).

`dataclass` é, mais ou menos, uma versão mutável de `NamedTuple`. "Mais ou menos" porque a semelhança é de propósito, não de implementação: um `NamedTuple` representa o dado como uma tupla por baixo dos panos, enquanto uma `dataclass` é uma classe Python normal para a qual o decorador gera automaticamente alguns métodos — `__init__`, `__repr__`, `__eq__` — que você teria que escrever à mão.

> **❗ Importante**
>
> `dataclass` existe desde o Python 3.7 — código que usa `@dataclass` não roda num interpretador mais antigo que isso.

A sintaxe é parecida com a de `NamedTuple`. Em vez de herdar de uma classe base, usa-se um decorador:

In [ ]:
import datetime
from dataclasses import dataclass

@dataclass
class StockPrice2:
    symbol: str
    date: datetime.date
    closing_price: float

    def is_high_tech(self) -> bool:
        """É uma classe, então também dá para acrescentar métodos"""
        return self.symbol in ['MSFT', 'GOOG', 'FB', 'AMZN', 'AAPL']

price2 = StockPrice2('MSFT', datetime.date(2018, 12, 14), 106.03)

assert price2.symbol == 'MSFT'
assert price2.closing_price == 106.03
assert price2.is_high_tech()

A diferença que importa: dá para modificar os valores de uma instância de `dataclass`.

In [ ]:
# desdobramento de ações
price2.closing_price /= 2
assert price2.closing_price == 106.03 / 2
price2.closing_price

Tentar a mesma coisa com a versão `NamedTuple` da seção anterior levantaria `AttributeError` — campos de `NamedTuple` não têm `setter`, porque por baixo continuam sendo posições de uma tupla.

Só que a mutabilidade reabre exatamente o problema que motivou trocar `dict` por `NamedTuple`, na seção anterior: como `StockPrice2` é uma classe normal, Python deixa você criar atributos novos nela em qualquer momento, sem checagem nenhuma.

In [ ]:
# é uma classe normal — então dá para acrescentar campos à vontade!
price2.cosing_price = 75   # opa

hasattr(price2, 'cosing_price'), hasattr(price2, 'closing_price')

O erro de digitação de duas seções atrás — `cosing_price` em vez de `closing_price` — volta a passar sem aviso nenhum. `dataclass` resolve o problema de desempenho e o de anotação de tipo do `dict`, mas não o de digitação: essa proteção específica é exclusiva do `NamedTuple`, porque tuplas não aceitam atributo novo de jeito nenhum.

> **💡 Dica — Na prática: o que se faz com isso**
>
> `dataclass` aparece bastante em código Python fora deste livro, sobretudo em lugares onde mutabilidade é mesmo necessária — um objeto que representa configuração sendo montada aos poucos, por exemplo, ou o estado interno de alguma coisa que muda ao longo do tempo. Onde o registro é fixo desde a criação — a leitura de uma linha de um CSV, o resultado de uma consulta —, `NamedTuple` continua sendo a escolha mais restritiva, e "mais restritivo" aqui é elogio: menos formas de errar.
>
> Bibliotecas de validação de dado, como `pydantic`, cobrem o caso em que nem `NamedTuple` nem `dataclass` bastam: uma classe com campos tipados que, ao ser construída, *verifica* os tipos de verdade — ao contrário da anotação pura de Python, que é só documentação (o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) mostrou isso: `add(10, "cinco")` com `a: int, b: int` não impede nada). `pydantic` é o que sustenta boa parte das APIs web modernas em Python — frameworks como FastAPI o usam para transformar um JSON de entrada em um objeto tipado, rejeitando a requisição se um campo vier com o tipo errado.

## Limpeza e Transformação

> **📌 Nota**
>
> Esta seção corresponde a *Cleaning and Munging*, do capítulo 10 de Grus (2019).

Dado do mundo real é **sujo**. Você geralmente vai ter que trabalhar nele antes de conseguir usá-lo: converter strings para `float` ou `int`, checar valores ausentes, tratar outliers e dado corrompido. O [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) já mostrou exemplos disso, ao ler linhas de arquivo que vêm sempre como texto puro.

Uma opção é fazer a conversão bem antes de usar o dado:

```python
closing_price = float(row[2])
```

Mas é provavelmente menos sujeito a erro fazer o *parsing* dentro de uma função que dá para testar isoladamente:

In [ ]:
import datetime
from typing import List, NamedTuple
from dateutil.parser import parse

class StockPrice(NamedTuple):
    symbol: str
    date: datetime.date
    closing_price: float

def parse_row(row: List[str]) -> StockPrice:
    symbol, date, closing_price = row
    return StockPrice(symbol=symbol,
                      date=parse(date).date(),
                      closing_price=float(closing_price))

# testa a função
stock = parse_row(["MSFT", "2018-12-14", "106.03"])

assert stock.symbol == "MSFT"
assert stock.date == datetime.date(2018, 12, 14)
assert stock.closing_price == 106.03

### E se o dado for ruim?

`parse_row` funciona, mas quebra ao primeiro valor inesperado — uma data mal formatada, um preço que não é número. Talvez você prefira receber `None` no lugar de um `ValueError` estourado no meio do programa:

In [ ]:
from typing import Optional
import re

def try_parse_row(row: List[str]) -> Optional[StockPrice]:
    symbol, date_, closing_price_ = row

    # o símbolo da ação deveria ser só letras maiúsculas
    if not re.match(r"^[A-Z]+$", symbol):
        return None

    try:
        date = parse(date_).date()
    except ValueError:
        return None

    try:
        closing_price = float(closing_price_)
    except ValueError:
        return None

    return StockPrice(symbol, date, closing_price)

# deveria devolver None para erros
assert try_parse_row(["MSFT0", "2018-12-14", "106.03"]) is None  # símbolo ruim
assert try_parse_row(["MSFT", "2018-12--14", "106.03"]) is None  # data ruim
assert try_parse_row(["MSFT", "2018-12-14", "x"]) is None        # preço ruim

# mas deveria devolver o mesmo de antes se o dado é bom
assert try_parse_row(["MSFT", "2018-12-14", "106.03"]) == stock

Agora podemos ler um arquivo real com dado sujo — `dados/comma_delimited_stock_prices.csv`, que tem exatamente esse problema:

In [ ]:
with open("dados/comma_delimited_stock_prices.csv") as f:
    print(f.read())

In [ ]:
import csv

data: List[StockPrice] = []

with open("dados/comma_delimited_stock_prices.csv") as f:
    reader = csv.reader(f)
    for row in reader:
        maybe_stock = try_parse_row(row)
        if maybe_stock is None:
            print(f"pulando linha inválida: {row}")
        else:
            data.append(maybe_stock)

len(data), data

Só uma linha é rejeitada: `MSFT,6/19/2014,n/a`, porque `"n/a"` não converte para `float`. As outras cinco entram — inclusive a da FB com data `6/20/3014`.

> **⚠️ Atenção — O erro que passa pelo filtro**
>
> Repare: `try_parse_row` não rejeitou a linha da FB, e o ano da data dela é **3014**. `"6/20/3014"` é uma string sintaticamente válida — `dateutil.parser.parse` converte sem reclamar, porque é um ano válido do ponto de vista do calendário, só que a mil anos de distância do resto do conjunto. O regex do símbolo não pega isso, e o `try/except ValueError` também não, porque não houve exceção nenhuma.
>
> É o mesmo tipo de outlier que a [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) existe para achar — por inspeção visual ou checagem *ad hoc* —, não algo que uma função de *parsing* bem escrita consegue capturar sozinha. Dado do mundo real tem casas decimais faltando, zeros a mais, erros de digitação e outros problemas do tipo — capturá-los é trabalho seu (talvez não seja oficialmente seu trabalho, mas quem mais vai fazer?).

E então é preciso decidir o que fazer com as linhas inválidas. Em geral, as três opções são: descartá-las, voltar à fonte e tentar corrigir o dado ausente ou ruim, ou não fazer nada e torcer. Se é uma linha ruim entre milhões, provavelmente está tudo bem ignorá-la. Mas se metade das suas linhas têm dado ruim, isso é algo que precisa ser corrigido.

> **💡 Dica — Na prática: `pandas`**
>
> `try_parse_row` — com seu regex, seus dois `try/except` e seu `None` como sinal de "descarte isto" — é o que `pd.read_csv(..., parse_dates=["date"], na_values=["n/a"])` faz por baixo, mais `pd.to_numeric(coluna, errors='coerce')` para o resto.
>
> O que isso esconde, e vale saber que está sendo escondido: a coerção do `pandas` é **silenciosa**. Onde `try_parse_row` devolve `None` e a linha é descartada com um aviso impresso, `errors='coerce'` não descarta nada — ele converte o valor problemático em `NaN` (ou `NaT`, para data), e a linha *sobrevive* no `DataFrame`. Ela não aparece como "pulando linha inválida" em lugar nenhum; ela só aparece depois, quando alguém calcula uma média sem checar `.isna()` antes, e o resultado sai errado sem erro nenhum. É o mesmo tipo de problema que a linha da FB com ano 3014 ilustrou acima — só que lá o dado passou pelo filtro por ser sintaticamente válido; aqui ele passa porque o `pandas` foi instruído a deixar passar.

## Manipulando Dados

> **📌 Nota**
>
> Esta seção corresponde a *Manipulating Data*, do capítulo 10 de Grus (2019).

Uma das habilidades mais importantes de quem trabalha com dado é **manipulá-lo**. É mais uma abordagem geral do que uma técnica específica — então esta seção percorre alguns exemplos, para dar a ideia.

Imagine que temos uma pilha de preços de ações como esta:

```python
data = [
    StockPrice(symbol='MSFT',
              date=datetime.date(2018, 12, 24),
              closing_price=106.03),
    # ...
]
```

e começamos a fazer perguntas sobre esse dado. Ao longo do caminho, vamos tentar notar padrões no que estamos fazendo e abstrair ferramentas que tornem a manipulação mais fácil.

Desta vez, em vez de um conjunto de brinquedo, vamos usar `dados/stocks.csv` inteiro — as 23.105 linhas de preços diários de quatro ações (AAPL, FB, GOOG, MSFT) já usadas no restante deste livro:

In [ ]:
import csv, re, datetime
from typing import Dict, List, NamedTuple, Optional
from dateutil.parser import parse

class StockPrice(NamedTuple):
    symbol: str
    date: datetime.date
    closing_price: float

def try_parse_row(row: List[str]) -> Optional[StockPrice]:
    symbol, date_, closing_price_ = row
    if not re.match(r"^[A-Z]+$", symbol):
        return None
    try:
        date = parse(date_).date()
    except ValueError:
        return None
    try:
        closing_price = float(closing_price_)
    except ValueError:
        return None
    return StockPrice(symbol, date, closing_price)

with open("dados/stocks.csv", "r") as f:
    reader = csv.DictReader(f)
    rows = [[row['Symbol'], row['Date'], row['Close']]
            for row in reader]

# garante que todas carregaram com sucesso
maybe_data = [try_parse_row(row) for row in rows]
assert maybe_data
assert all(sp is not None for sp in maybe_data)

data = [sp for sp in maybe_data if sp is not None]
len(data), sorted(set(sp.symbol for sp in data))

> **📌 Nota**
>
> Diferente do CSV de brinquedo da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-limpeza-e-transformacao.html), aqui as 23.105 linhas passam **todas** por `try_parse_row` sem cair fora — este arquivo não tem o mesmo tipo de linha corrompida. É o que o `assert all(...)` acima confirma: se uma linha só tivesse voltado `None`, ele estouraria.

Suponha que queremos saber o maior preço de fechamento já registrado para a AAPL. Podemos dividir isso em passos concretos:

1. Restringir às linhas da AAPL.
2. Pegar o `closing_price` de cada linha.
3. Tirar o `max` desses preços.

Uma compreensão faz as três coisas de uma vez:

In [ ]:
max_aapl_price = max(stock_price.closing_price
                     for stock_price in data
                     if stock_price.symbol == "AAPL")

max_aapl_price

Mais geralmente, podemos querer saber o maior preço de fechamento de cada ação no conjunto. Uma forma de fazer isso:

1. Criar um `dict` para guardar o maior preço de cada símbolo (usando um `defaultdict` que devolve menos infinito para símbolos ainda não vistos, já que qualquer preço será maior que isso).
2. Percorrer os dados, atualizando esse `dict`.

In [ ]:
from collections import defaultdict

max_prices: Dict[str, float] = defaultdict(lambda: float('-inf'))

for sp in data:
    symbol, closing_price = sp.symbol, sp.closing_price
    if closing_price > max_prices[symbol]:
        max_prices[symbol] = closing_price

dict(sorted(max_prices.items()))

Agora podemos começar a fazer perguntas mais complicadas, como quais foram as maiores e as menores variações percentuais em um único dia no conjunto de dados. A variação percentual é `preço_hoje / preço_ontem - 1`, o que significa que precisamos de alguma forma de associar o preço de hoje ao de ontem. Uma abordagem é agrupar os preços por símbolo e, dentro de cada grupo:

1. Ordenar os preços por data.
2. Usar `zip` para formar pares (anterior, atual).
3. Transformar os pares em novas linhas de "variação percentual".

Vamos começar agrupando os preços por símbolo:

In [ ]:
prices: Dict[str, List[StockPrice]] = defaultdict(list)

for sp in data:
    prices[sp.symbol].append(sp)

Como os preços são tuplas, eles são ordenados pelos campos, na ordem: primeiro por símbolo, depois por data, depois por preço. Isso significa que, se temos preços todos com o mesmo símbolo, `sort` os ordena por data (e depois por preço, o que não muda nada, já que só há um preço por data) — que é o que queremos:

In [ ]:
# ordena os preços por data
prices = {symbol: sorted(symbol_prices)
          for symbol, symbol_prices in prices.items()}

o que podemos usar para calcular uma sequência de variações dia a dia:

In [ ]:
def pct_change(yesterday: StockPrice, today: StockPrice) -> float:
    return today.closing_price / yesterday.closing_price - 1

class DailyChange(NamedTuple):
    symbol: str
    date: datetime.date
    pct_change: float

def day_over_day_changes(prices: List[StockPrice]) -> List[DailyChange]:
    """Assume que os preços são de uma única ação e estão em ordem"""
    return [DailyChange(symbol=today.symbol,
                        date=today.date,
                        pct_change=pct_change(yesterday, today))
            for yesterday, today in zip(prices, prices[1:])]

`zip(prices, prices[1:])` é o mesmo truque de percorrer duas sequências em paralelo que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) apresentou: aqui, desalinhar uma lista de si mesma por uma posição transforma a sequência inteira numa sequência de pares (ontem, hoje), prontos para `pct_change`.

e depois juntar todas:

In [ ]:
all_changes = [change
              for symbol_prices in prices.values()
              for change in day_over_day_changes(symbol_prices)]

len(all_changes)

A partir daqui é fácil achar a maior e a menor:

In [ ]:
max_change = max(all_changes, key=lambda change: change.pct_change)
min_change = min(all_changes, key=lambda change: change.pct_change)

max_change, min_change

O maior salto de um dia para o outro foi a AAPL, em 6 de agosto de 1997 — mais de 33% em um único pregão. A maior queda foi também a AAPL, em 29 de setembro de 2000 — quase 52% em um dia. As duas datas batem com eventos reais e documentados: o investimento de US$ 150 milhões da Microsoft na Apple, anunciado naquele agosto de 1997, e o alerta de resultados abaixo do esperado que a Apple divulgou em setembro de 2000.

Podemos agora usar esse conjunto `all_changes` para descobrir qual é o melhor mês para investir em ações de tecnologia. Basta olhar a variação diária média por mês:

In [ ]:
changes_by_month: Dict[int, List[DailyChange]] = {month: [] for month in range(1, 13)}

for change in all_changes:
    changes_by_month[change.date.month].append(change)

avg_daily_change = {
    month: sum(change.pct_change for change in changes) / len(changes)
    for month, changes in changes_by_month.items()
}

{mes: round(v, 5) for mes, v in avg_daily_change.items()}

> **📌 Nota**
>
> Repare na anotação de `changes_by_month`: `Dict[int, List[DailyChange]]` é o tipo que o valor de fato tem. Se estivesse escrito `List[DailyChange]` ali — uma `List` recebendo um `dict` —, o código rodaria exatamente igual, porque **anotações de tipo não são verificadas em tempo de execução**, como o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) já avisou. Um verificador como `mypy` acusaria o descompasso na hora; o interpretador Python, sozinho, deixa passar — a anotação nunca influencia o que é executado, então um descompasso desses sobrevive indefinidamente num código que roda todo dia.

In [ ]:
melhor_mes = max(avg_daily_change, key=lambda mes: avg_daily_change[mes])
assert melhor_mes == 10   # outubro é o melhor mês

melhor_mes, avg_daily_change[melhor_mes]

Outubro é o melhor mês para investir, com uma variação diária média de cerca de 0,29% — número que vale hesitar antes de levar a sério: são apenas quatro ações e quase quatro décadas de dados misturados numa única média por mês, sem nenhum teste de significância por trás. É o tipo de padrão fácil de achar e fácil demais de acreditar.

Faremos esse tipo de manipulação pelo livro inteiro, em geral sem chamar muita atenção para ela.

> **💡 Dica — Na prática: `pandas`**
>
> Tudo que esta seção fez à mão tem uma linha de `pandas` correspondente: `df.groupby('Symbol')['Close'].max()` no lugar do `defaultdict` que acha o maior preço por símbolo; `df.groupby('Symbol')['Close'].pct_change()` no lugar de `day_over_day_changes`; `df.resample('ME')` para agregações por mês, no lugar do `dict` `changes_by_month` construído à mão — e repare no `'ME'`, de *month end*: o `'M'` que a maior parte do material antigo usa foi depreciado no `pandas` 2.2.
>
> O que isso esconde, e vale saber que está sendo escondido: `.pct_change()` não ordena nada — ela assume que as linhas já chegam ordenadas e contíguas, exatamente a etapa `sorted(symbol_prices)` que esta seção gastou um parágrafo justificando antes de calcular `day_over_day_changes`. Esqueça o `sort_values('Date')` antes de agrupar e o `pandas` não levanta erro nenhum: `.pct_change()` roda normalmente e devolve, em silêncio, variações entre dias que não são consecutivos — uma "variação diária" que na verdade compara terça com sexta.

## Reescalonamento

> **📌 Nota**
>
> Esta seção corresponde a *Rescaling*, do capítulo 10 de Grus (2019).

Muitas técnicas são sensíveis à **escala** dos dados. Imagine um conjunto com altura e peso de centenas de cientistas de dados, e que você está tentando identificar agrupamentos de porte físico.

Intuitivamente, gostaríamos que agrupamentos representassem pontos próximos entre si, o que significa que precisamos de alguma noção de distância entre pontos. Já temos uma função de distância euclidiana — do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html) —, então uma abordagem natural é tratar pares (altura, peso) como pontos num espaço bidimensional. Considere as pessoas listadas abaixo:

| Pessoa | Altura (polegadas) | Altura (centímetros) | Peso (libras) |
|---|---|---|---|
| A | 63 | 160 | 150 |
| B | 67 | 170,2 | 160 |
| C | 70 | 177,8 | 171 |

Se medirmos altura em polegadas, o vizinho mais próximo de B é A:

In [ ]:
from scratch.linear_algebra import distance

a_to_b = distance([63, 150], [67, 160])
a_to_c = distance([63, 150], [70, 171])
b_to_c = distance([67, 160], [70, 171])

round(a_to_b, 2), round(a_to_c, 2), round(b_to_c, 2)

Só que, se medirmos altura em centímetros, o vizinho mais próximo de B passa a ser C:

In [ ]:
a_to_b = distance([160, 150], [170.2, 160])
a_to_c = distance([160, 150], [177.8, 171])
b_to_c = distance([170.2, 160], [177.8, 171])

round(a_to_b, 2), round(a_to_c, 2), round(b_to_c, 2)

> **⚠️ Atenção**
>
> Nada mudou nos dados — A, B e C continuam as mesmas três pessoas. Só a **unidade** de uma das duas dimensões mudou, e isso bastou para inverter qual ponto é "mais próximo" de qual. Qualquer técnica baseada em distância — vizinhos mais próximos, incluído — está exposta a esse problema sempre que as dimensões não são comparáveis entre si.

Obviamente é um problema se trocar a unidade pode mudar resultados desse jeito. Por essa razão, quando as dimensões não são comparáveis entre si, às vezes vamos **reescalonar** nossos dados, de modo que cada dimensão tenha média 0 e desvio padrão 1. Isso efetivamente elimina as unidades, convertendo cada dimensão em "desvios padrão a partir da média".

Para começar, precisamos calcular a média e o desvio padrão de cada posição:

In [ ]:
from scratch.statistics import standard_deviation
import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
from typing import List, Tuple
from scratch.linear_algebra import Vector, vector_mean

def scale(data: List[Vector]) -> Tuple[Vector, Vector]:
    """devolve a média e o desvio padrão de cada posição"""
    dim = len(data[0])

    means = vector_mean(data)
    stdevs = [standard_deviation([vector[i] for vector in data])
              for i in range(dim)]

    return means, stdevs

vectors = [[-3, -1, 1], [-1, 0, 1], [1, 1, 1]]
means, stdevs = scale(vectors)

assert means == [-1, 0, 1]
assert stdevs == [2, 1, 0]
means, stdevs

Repare na terceira posição: todo vetor tem 1 ali, então o desvio padrão é 0 — não há variação nenhuma para reescalonar.

Com `means` e `stdevs`, podemos criar um novo conjunto de dados:

In [ ]:
def rescale(data: List[Vector]) -> List[Vector]:
    """
    Reescalona os dados de entrada para que cada posição tenha
    média 0 e desvio padrão 1. (Deixa uma posição como está se
    o desvio padrão dela é 0.)
    """
    dim = len(data[0])
    means, stdevs = scale(data)

    # copia cada vetor
    rescaled = [v[:] for v in data]

    for v in rescaled:
        for i in range(dim):
            if stdevs[i] > 0:
                v[i] = (v[i] - means[i]) / stdevs[i]

    return rescaled

rescale(vectors)

E, claro, vale testar que `rescale` faz o que achamos que faz:

In [ ]:
means, stdevs = scale(rescale(vectors))
assert means == [0, 0, 1]
assert stdevs == [1, 1, 0]

A terceira posição continua com desvio padrão 0 — `rescale` deliberadamente deixa em paz qualquer dimensão sem variação, em vez de dividir por zero.

> **🔷 Conceito**
>
> Como sempre, é preciso usar julgamento. Se você pegasse um conjunto de dados grande e o filtrasse para conter só pessoas com altura entre 69,5 e 70,5 polegadas, é bem provável que a variação restante naquela dimensão seja só ruído — e talvez você não quisesse colocar o desvio padrão dela em pé de igualdade com o das outras dimensões. Reescalonar não é neutro: é uma decisão sobre o que conta como sinal.

> **💡 Dica — Na prática: `scikit-learn`**
>
> `scale` e `rescale` juntos são exatamente o `StandardScaler` do `scikit-learn`:
>
> ```python
> from sklearn.preprocessing import StandardScaler
>
> escalonador = StandardScaler()
> dados_reescalonados = escalonador.fit_transform(dados)
> ```
>
> `fit` calcula média e desvio padrão de cada coluna (o que você fez em `scale`); `transform` aplica a fórmula `(x - média) / desvio` (o que você fez em `rescale`); `fit_transform` faz as duas coisas em sequência. A separação entre `fit` e `transform` importa na prática: você ajusta o escalonador **só** nos dados de treino, e aplica a mesma transformação — as mesmas médias e desvios, calculados no treino — aos dados de teste. Se você recalculasse média e desvio em cima do conjunto de teste, estaria vazando informação dele para dentro do seu pipeline antes mesmo de avaliar o modelo.
>
> O `scikit-learn` também tem `MinMaxScaler`, que reescalona para um intervalo fixo (geralmente `[0, 1]`) em vez de média 0 e desvio padrão 1 — útil quando a distribuição dos dados está longe de normal e desvio padrão não é uma unidade natural para eles.

## Um Parêntese: tqdm

> **📌 Nota**
>
> Esta seção corresponde a *An Aside: tqdm*, do capítulo 10 de Grus (2019).

Boa parte do que vem daqui para frente neste livro envolve cálculos que demoram — laços com dezenas de milhares de iterações, gradiente descendente rodando por centenas de passos. Quando um cálculo desses está em andamento, é bom saber que ele está de fato progredindo, e ter alguma ideia de quanto falta.

Uma forma de fazer isso é a biblioteca `tqdm`, que gera barras de progresso configuráveis.

Há só duas coisas que você realmente precisa saber sobre ela. A primeira é que envolver um iterável em `tqdm.tqdm` produz uma barra de progresso:

In [ ]:
import tqdm
import random

random.seed(0)

for i in tqdm.tqdm(range(100)):
    # faz algo lento
    _ = [random.random() for _ in range(1_000_000)]

O que aparece no terminal, ao vivo, é uma linha que se atualiza a cada iteração — algo como:

```
 56%|███████████████                     | 56/100 [00:08<00:06,  6.49it/s]
```

Em particular, ela mostra que fração do laço já terminou, há quanto tempo está rodando, e quanto tempo ainda deve levar (a não ser que você esteja envolvendo um gerador, caso em que `tqdm` não tem como saber o tamanho total). Como o laço acima percorre um `range`, `tqdm.tqdm` sabe o total de antemão e consegue estimar o tempo restante.

> **🟩 Exemplo**
>
> `tqdm` escreve a barra no fluxo de erro padrão (`stderr`), não na saída padrão, e reescreve sempre a mesma linha. É por isso que a barra se atualiza no lugar em vez de encher a tela — e também por que ela não vai junto quando você redireciona a saída padrão do programa para um arquivo: a barra continua no terminal, separada do resultado.

A segunda coisa é que dá para configurar a descrição da barra enquanto ela roda. Para isso, você precisa capturar o iterador do `tqdm` com um `with`. No caso de estarmos apenas envolvendo `range`, dá para usar diretamente `tqdm.trange`:

In [ ]:
from typing import List

def primes_up_to(n: int) -> List[int]:
    primes = [2]

    with tqdm.trange(3, n) as t:
        for i in t:
            # i é primo se nenhum primo menor o divide
            i_is_prime = not any(i % p == 0 for p in primes)
            if i_is_prime:
                primes.append(i)

            t.set_description(f"{len(primes)} primos")

    return primes

my_primes = primes_up_to(100_000)

len(my_primes)

Isso acrescenta uma descrição como a seguinte, com um contador que se atualiza conforme novos primos são encontrados:

```
5088 primos:  50%|████████               | 49529/99997 [00:03<00:03, 15905.90it/s]
```

> **🟩 Exemplo**
>
> `primes_up_to` não é o algoritmo mais rápido possível para achar primos — para cada `i`, ele testa divisibilidade contra **todos** os primos já encontrados, não só os menores que $\sqrt{i}$. Isso é proposital: o ponto aqui não é otimizar a busca por primos, é ter um laço genuinamente lento para demonstrar `tqdm` em cima de algo real, em vez de um `time.sleep` artificial.

Usar `tqdm` vai, de vez em quando, deixar seu código um pouco instável — às vezes a tela redesenha mal, às vezes o laço trava de verdade. E se você acidentalmente aninhar um laço `tqdm` dentro de outro laço `tqdm`, coisas estranhas podem acontecer. Ainda assim, os benefícios costumam superar essas desvantagens, então vamos usá-lo sempre que tivermos cálculos que demoram — a começar pela próxima seção, onde `tqdm.trange` acompanha os passos do gradiente descendente ajustando a primeira componente principal.

> **💡 Dica — Na prática: o que se faz com isso**
>
> `tqdm` não é algo que uma biblioteca de modelagem substitui — ele é ferramenta de operação, não de cálculo, e continua sendo a escolha usada em produção para acompanhar laços Python comuns. O que muda fora deste livro é a integração: `tqdm.pandas()` registra uma barra de progresso para `DataFrame.progress_apply`, e frameworks de treinamento de rede neural (o assunto dos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) deste livro) costumam ter uma barra de progresso embutida, no mesmo espírito, para acompanhar epoch a epoch de treino.

## Redução de Dimensionalidade

> **📌 Nota**
>
> Esta seção corresponde a *Dimensionality Reduction*, do capítulo 10 de Grus (2019).

Às vezes as dimensões "de verdade" (ou úteis) de um conjunto de dados não correspondem às dimensões que ele tem. Considere o conjunto de dados a seguir:

In [ ]:
pca_data = [
[20.9666776351559,-13.1138080189357],
[22.7719907680008,-19.8890894944696],
[25.6687103160153,-11.9956004517219],
[18.0019794950564,-18.1989191165133],
[21.3967402102156,-10.8893126308196],
[0.443696899177716,-19.7221132386308],
[29.9198322142127,-14.0958668502427],
[19.0805843080126,-13.7888747608312],
[16.4685063521314,-11.2612927034291],
[21.4597664701884,-12.4740034586705],
[3.87655283720532,-17.575162461771],
[34.5713920556787,-10.705185165378],
[13.3732115747722,-16.7270274494424],
[20.7281704141919,-8.81165591556553],
[24.839851437942,-12.1240962157419],
[20.3019544741252,-12.8725060780898],
[21.9021426929599,-17.3225432396452],
[23.2285885715486,-12.2676568419045],
[28.5749111681851,-13.2616470619453],
[29.2957424128701,-14.6299928678996],
[15.2495527798625,-18.4649714274207],
[26.5567257400476,-9.19794350561966],
[30.1934232346361,-12.6272709845971],
[36.8267446011057,-7.25409849336718],
[32.157416823084,-10.4729534347553],
[5.85964365291694,-22.6573731626132],
[25.7426190674693,-14.8055803854566],
[16.237602636139,-16.5920595763719],
[14.7408608850568,-20.0537715298403],
[6.85907008242544,-18.3965586884781],
[26.5918329233128,-8.92664811750842],
[-11.2216019958228,-27.0519081982856],
[8.93593745011035,-20.8261235122575],
[24.4481258671796,-18.0324012215159],
[2.82048515404903,-22.4208457598703],
[30.8803004755948,-11.455358009593],
[15.4586738236098,-11.1242825084309],
[28.5332537090494,-14.7898744423126],
[40.4830293441052,-2.41946428697183],
[15.7563759125684,-13.5771266003795],
[19.3635588851727,-20.6224770470434],
[13.4212840786467,-19.0238227375766],
[7.77570680426702,-16.6385739839089],
[21.4865983854408,-15.290799330002],
[12.6392705930724,-23.6433305964301],
[12.4746151388128,-17.9720169566614],
[23.4572410437998,-14.602080545086],
[13.6878189833565,-18.9687408182414],
[15.4077465943441,-14.5352487124086],
[20.3356581548895,-10.0883159703702],
[20.7093833689359,-12.6939091236766],
[11.1032293684441,-14.1383848928755],
[17.5048321498308,-9.2338593361801],
[16.3303688220188,-15.1054735529158],
[26.6929062710726,-13.306030567991],
[34.4985678099711,-9.86199941278607],
[39.1374291499406,-10.5621430853401],
[21.9088956482146,-9.95198845621849],
[22.2367457578087,-17.2200123442707],
[10.0032784145577,-19.3557700653426],
[14.045833906665,-15.871937521131],
[15.5640911917607,-18.3396956121887],
[24.4771926581586,-14.8715313479137],
[26.533415556629,-14.693883922494],
[12.8722580202544,-21.2750596021509],
[24.4768291376862,-15.9592080959207],
[18.2230748567433,-14.6541444069985],
[4.1902148367447,-20.6144032528762],
[12.4332594022086,-16.6079789231489],
[20.5483758651873,-18.8512560786321],
[17.8180560451358,-12.5451990696752],
[11.0071081078049,-20.3938092335862],
[8.30560561422449,-22.9503944138682],
[33.9857852657284,-4.8371294974382],
[17.4376502239652,-14.5095976075022],
[29.0379635148943,-14.8461553663227],
[29.1344666599319,-7.70862921632672],
[32.9730697624544,-15.5839178785654],
[13.4211493998212,-20.150199857584],
[11.380538260355,-12.8619410359766],
[28.672631499186,-8.51866271785711],
[16.4296061111902,-23.3326051279759],
[25.7168371582585,-13.8899296143829],
[13.3185154732595,-17.8959160024249],
[3.60832478605376,-25.4023343597712],
[39.5445949652652,-11.466377647931],
[25.1693484426101,-12.2752652925707],
[25.2884257196471,-7.06710309184533],
[6.77665715793125,-22.3947299635571],
[20.1844223778907,-16.0427471125407],
[25.5506805272535,-9.33856532270204],
[25.1495682602477,-7.17350567090738],
[15.6978431006492,-17.5979197162642],
[37.42780451491,-10.843637288504],
[22.974620174842,-10.6171162611686],
[34.6327117468934,-9.26182440487384],
[34.7042513789061,-6.9630753351114],
[15.6563953929008,-17.2196961218915],
[25.2049825789225,-14.1592086208169]
]
len(pca_data)

Grande parte da variação dos dados parece estar ao longo de uma única dimensão, que não corresponde nem ao eixo $x$ nem ao eixo $y$:

In [ ]:
# Figura: Dados com os eixos \"errados\
from matplotlib import pyplot as plt

xs = [p[0] for p in pca_data]
ys = [p[1] for p in pca_data]
plt.scatter(xs, ys, s=15, color="darkblue")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados com os eixos \"errados\"")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

Quando isso acontece, dá para usar uma técnica chamada **análise de componentes principais** (PCA, na sigla em inglês) para extrair uma ou mais dimensões que capturem o máximo possível da variação nos dados.

> **❗ Importante**
>
> Na prática, você não usaria essa técnica num conjunto de dados tão baixo-dimensional quanto este. Redução de dimensionalidade é útil principalmente quando o conjunto tem um número grande de dimensões e você quer achar um pequeno subconjunto delas que capture a maior parte da variação. Infelizmente, esse caso é difícil de ilustrar num formato de livro bidimensional — por isso o exemplo aqui tem só duas dimensões, mesmo a técnica sendo pensada para muitas.

> **📌 Nota**
>
> Esta seção depende do gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) — especificamente de `gradient_step`, a função de cinco linhas construída lá que treina praticamente todo modelo deste livro. Aqui ela não ajusta parâmetro de modelo nenhum; ajusta a **direção** que melhor explica a variação dos dados. Se `gradient_step` não estiver familiar, vale revisitar aquele capítulo antes de continuar.

### Centralizando os dados

Como primeiro passo, precisamos transladar os dados para que cada dimensão tenha média 0:

In [ ]:
from typing import List
from scratch.linear_algebra import Vector, subtract, vector_mean

def de_mean(data: List[Vector]) -> List[Vector]:
    """Recentra os dados para que cada dimensão tenha média 0"""
    mean = vector_mean(data)
    return [subtract(vector, mean) for vector in data]

de_meaned = de_mean(pca_data)

(Se não fizermos isso, é bem provável que nossas técnicas identifiquem a própria média em vez da variação nos dados.)

Centralizar é a metade da [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) que sobrou: mesma translação, sem dividir pelo desvio padrão. Por que só a metade — e quando a outra também precisa vir junto — é o que a "Na prática" desta seção fecha.

In [ ]:
# Figura: Dados depois de remover a média
xs2 = [p[0] for p in de_meaned]
ys2 = [p[1] for p in de_meaned]
plt.scatter(xs2, ys2, s=15, color="darkblue")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados depois de remover a média")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

A forma da nuvem de pontos não mudou nada — só o centro dela, que agora é a origem.

### A direção de maior variância

Agora, dada uma matriz $X$ de dados centralizados, podemos perguntar qual é a direção que captura a maior variância nos dados.

Por que variância, e não alguma outra medida de espalhamento? Porque projetar os dados numa única direção e depois tentar reconstruí-los a partir só dessa projeção perde exatamente a variação que ficou de fora dela — e a direção que **maximiza** a variância capturada é a mesma que **minimiza** o quanto se perde ao descartar o resto. São o mesmo problema, visto de dois lados; resolvemos pelo lado que dá uma função mais fácil de diferenciar.

> **🔷 Conceito**
>
> A direção de maior variância é a direção em que os dados menos se parecem com um ponto só — o que sobra de informação depois de projetar tudo nela e aceitar não guardar mais nada. Reduzir dimensionalidade é escolher, de propósito, o que descartar primeiro: o que varia menos.

Vale fixar também os dois sentidos da palavra **componente**, porque o restante desta seção usa os dois: primeiro, uma **componente principal** é uma *direção* — um vetor de magnitude 1, como o `fpc` que `first_principal_component` devolve daqui a pouco. Depois, quando um ponto é projetado nessa direção, o número que sai da projeção — o `dot(v, w)` que a seção "Projetando e removendo", adiante, vai calcular — também é chamado de "a componente" daquele ponto. A primeira é uma seta no espaço original; a segunda é uma coordenada nova, medida ao longo dela.

Especificamente, dada uma direção $d$ (um vetor de magnitude 1), a projeção de cada linha $x$ da matriz sobre $d$ tem comprimento `dot(x, d)` — é o quanto daquele ponto "cabe" naquela direção. E todo vetor $w$ não nulo determina uma direção, se o reescalonarmos para ter magnitude 1:

In [ ]:
from scratch.linear_algebra import magnitude

def direction(w: Vector) -> Vector:
    mag = magnitude(w)
    return [w_i / mag for w_i in w]

Portanto, dado um vetor $w$ não nulo, podemos calcular a variância do nosso conjunto de dados na direção determinada por $w$:

In [ ]:
from scratch.linear_algebra import dot

def directional_variance(data: List[Vector], w: Vector) -> float:
    """Devolve a variância de x na direção de w"""
    w_dir = direction(w)
    return sum(dot(v, w_dir) ** 2 for v in data)

> **📌 Nota**
>
> Apesar do nome e do *docstring*, `directional_variance` não divide por $n$: é uma **soma de quadrados**, não a variância no sentido em que o resto deste livro usa a palavra. Como os dados já passaram por `de_mean`, essa soma é proporcional à variância de verdade — dividir por $n$ mudaria a escala do número, mas não a direção $w$ que o maximiza, que é tudo que interessa aqui. Se você for comparar este valor contra a fórmula de variância que já conhece, é este o motivo do descompasso.

Gostaríamos de achar a direção que maximiza essa variância. Podemos fazer isso com gradiente descendente, assim que tivermos a função de gradiente:

In [ ]:
def directional_variance_gradient(data: List[Vector], w: Vector) -> Vector:
    """O gradiente da variância direcional em relação a w"""
    w_dir = direction(w)
    return [sum(2 * dot(v, w_dir) * v[i] for v in data)
            for i in range(len(w))]

> **📌 Nota**
>
> Essa fórmula não é derivada aqui — vale aceitá-la como dada, mas com a razão de ela ser legítima: `directional_variance` soma termos $(v \cdot w_{dir})^2$, e a derivada de um quadrado como esse em relação a cada coordenada de $w$ é o dobro do produto interno vezes a coordenada correspondente de $v$ — exatamente o `2 * dot(v, w_dir) * v[i]` acima, somado sobre todos os pontos. É cálculo de uma variável aplicado termo a termo, não uma fórmula nova.

E agora a primeira componente principal é justamente a direção que maximiza a função `directional_variance`:

In [ ]:
import tqdm
from scratch.gradient_descent import gradient_step

def first_principal_component(data: List[Vector],
                              n: int = 100,
                              step_size: float = 0.1) -> Vector:
    # começa com um palpite aleatório
    guess = [1.0 for _ in data[0]]

    with tqdm.trange(n) as t:
        for _ in t:
            dv = directional_variance(data, guess)
            gradient = directional_variance_gradient(data, guess)
            guess = gradient_step(guess, gradient, step_size)
            t.set_description(f"dv: {dv:.3f}")

    return direction(guess)

fpc = first_principal_component(de_meaned)
assert 0.923 < fpc[0] < 0.925
assert 0.382 < fpc[1] < 0.384
fpc

> **❗ Importante**
>
> Repare no sinal: `step_size` aqui é **positivo**. Todo o resto deste livro passa passo negativo para `gradient_step` — para andar *contra* o gradiente e minimizar algum erro, como o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) explica ao apresentar essa mesma função. Aqui não há erro para minimizar: queremos **maximizar** a variância direcional, então andamos **com** o gradiente. É o único uso ascendente do livro inteiro — e é o mesmo `gradient_step` de cinco linhas do Capítulo 5, usado ao contrário.

> **📌 Nota**
>
> O comentário diz "palpite aleatório", mas repare no código: `guess` começa sempre em `[1.0, 1.0]`. Não há aleatoriedade nenhuma aqui — o gradiente parte sempre do mesmo lugar e segue sempre o mesmo caminho, então o resultado é determinístico e os dois `assert` acima valem em qualquer execução, sem precisar de `random.seed`. Uma inicialização genuinamente aleatória — como a que o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) usa para os pesos de uma rede neural — exigiria a semente para valer o mesmo.

No conjunto de dados centralizado, isso devolve a direção aproximadamente $(0{,}924,\ 0{,}383)$, que de fato parece capturar o eixo principal ao longo do qual nossos dados variam:

In [ ]:
# Figura: Primeira componente principal
escala = 30
plt.scatter(xs2, ys2, s=15, color="darkblue")
plt.plot([-escala * fpc[0], escala * fpc[0]],
        [-escala * fpc[1], escala * fpc[1]],
        color="black", linewidth=1.5)
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Primeira componente principal")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

### Projetando e removendo

Uma vez achada a direção que é a primeira componente principal, podemos projetar nossos dados nela para achar os valores dessa componente:

In [ ]:
from scratch.linear_algebra import scalar_multiply

def project(v: Vector, w: Vector) -> Vector:
    """devolve a projeção de v na direção w"""
    projection_length = dot(v, w)
    return scalar_multiply(projection_length, w)

Se quisermos achar componentes adicionais, primeiro removemos as projeções dos dados:

In [ ]:
def remove_projection_from_vector(v: Vector, w: Vector) -> Vector:
    """projeta v em w e subtrai o resultado de v"""
    return subtract(v, project(v, w))

def remove_projection(data: List[Vector], w: Vector) -> List[Vector]:
    return [remove_projection_from_vector(v, w) for v in data]

residual = remove_projection(de_meaned, fpc)

Como este conjunto de exemplo é só bidimensional, depois de removermos a primeira componente o que sobra é efetivamente unidimensional:

In [ ]:
# Figura: Dados depois de remover a primeira componente principal
xs3 = [p[0] for p in residual]
ys3 = [p[1] for p in residual]
plt.scatter(xs3, ys3, s=15, color="darkblue")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados depois de remover a primeira componente principal")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

Os pontos ainda têm duas coordenadas, mas em duas dimensões, remover a projeção sobre uma direção deixa exatamente a direção ortogonal — daí a reta perfeita: toda a variação que sobrou está numa única direção, a segunda componente principal.

### Muitas componentes

Num conjunto de dados de maior dimensão, podemos achar quantas componentes quisermos iterando o processo: ache a direção de maior variância, remova a projeção nela, repita no que sobrou.

In [ ]:
def pca(data: List[Vector], num_components: int) -> List[Vector]:
    components: List[Vector] = []
    for _ in range(num_components):
        component = first_principal_component(data)
        components.append(component)
        data = remove_projection(data, component)

    return components

E então podemos **transformar** nossos dados para o espaço de dimensão menor gerado pelas componentes:

In [ ]:
def transform_vector(v: Vector, components: List[Vector]) -> Vector:
    return [dot(v, w) for w in components]

def transform(data: List[Vector], components: List[Vector]) -> List[Vector]:
    return [transform_vector(v, components) for v in data]

componentes = pca(de_meaned, 2)
transformado = transform(de_meaned, componentes)
transformado[:5]

Com as duas componentes deste exemplo bidimensional, `transform` só reexpressa cada ponto num novo par de eixos — não reduz nada. O ganho aparece quando `num_components` é bem menor que o número de dimensões originais: um conjunto de centenas de colunas pode virar um punhado de componentes que ainda capturam a maior parte da variação. É o problema que ficou em aberto na [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html): a matriz de dispersão é uma ferramenta de triagem que não escala além de um punhado de dimensões. PCA é a resposta para quando há dimensões demais até para desenhar essa matriz.

Essa técnica é valiosa por duas razões. Primeiro, ela pode ajudar a limpar os dados, eliminando dimensões de ruído e consolidando dimensões fortemente correlacionadas. Segundo, depois de extrair uma representação de baixa dimensão dos dados, dá para usar uma variedade de técnicas que não funcionam tão bem em dados de alta dimensão — o assunto da [seção sobre a maldição da dimensionalidade](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-a-maldicao-da-dimensionalidade.html). Ela também é a resposta que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html), sobre seleção de atributos, prometia: uma forma de **remover** atributos em vez de criá-los, quando o número deles é grande demais.

Ao mesmo tempo, embora essa técnica possa ajudar a construir modelos melhores, ela também pode tornar esses modelos mais difíceis de interpretar. É fácil entender uma conclusão como "cada ano a mais de experiência aumenta o salário médio em 10 mil reais". É bem mais difícil dar sentido a "cada aumento de 0,1 na terceira componente principal aumenta o salário médio em 10 mil reais".

É aqui que se fecha a distância anunciada na abertura deste capítulo. O dado que chegou pelos canos do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) — torto, incompleto, cheio de outliers e datas impossíveis — passou por sete seções e chega a este ponto explorado, tipado, limpo, agregado, em escala comum e, agora, com dimensões a menos. É exatamente o formato que a máquina de gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) espera receber.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.decomposition import PCA
>
> modelo = PCA(n_components=2)
> modelo.fit(dados)
> dados_transformados = modelo.transform(dados)
>
> modelo.explained_variance_ratio_  # quanto cada componente explica
> ```
>
> A diferença mais importante não é de interface, é de método. O que você construiu acima acha uma componente de cada vez, por gradiente descendente, e remove a projeção antes de achar a próxima — um processo iterativo e aproximado. O `scikit-learn` resolve para **todas** as componentes de uma vez, por decomposição em valores singulares (SVD), o que é ao mesmo tempo mais rápido e numericamente mais estável — sem depender de tamanho de passo, número de iterações ou um palpite inicial, como o seu `first_principal_component` depende.
>
> Duas pegadinhas que `PCA` do `scikit-learn` não avisa sozinho: primeiro, `fit` centraliza os dados automaticamente (o equivalente ao seu `de_mean`), mas **não** reescalona por desvio padrão — se as dimensões têm unidades muito diferentes, é preciso rodar `StandardScaler` (a [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html)) antes de `PCA`, senão a dimensão de maior variância numérica domina as componentes só por causa da escala, não porque carregue mais informação de verdade. Segundo, o sinal de cada componente é **matematicamente** arbitrário: $(0{,}924,\ 0{,}383)$ e $(-0{,}924,\ -0{,}383)$ descrevem a mesma direção, e nenhuma das duas está "mais certa" que a outra — a sua implementação poderia devolver qualquer uma, dependendo do palpite inicial. O `scikit-learn`, porém, **não** deixa isso ao acaso: ele passa o resultado por `svd_flip`, que fixa o sinal por convenção. Rodando os quatro solvers de `PCA` sobre estes mesmos dados, todos devolvem $(0{,}924,\ 0{,}383)$, sempre.
>
> Isso é útil e é exatamente o tipo de decisão que uma biblioteca toma por você sem avisar. Útil, porque duas execuções concordam e o seu gráfico não vira de cabeça para baixo entre uma rodada e outra. Silencioso, porque em lugar nenhum da saída está escrito que houve uma escolha — e se você comparar as suas componentes com as de outra ferramenta, que adote a convenção oposta, os sinais vão discordar sem que nada esteja errado.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 10 de Grus (2019) sugere:

- [pandas](https://pandas.pydata.org/) é, de novo, a resposta de fora deste livro. Já foi mencionado no [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) para importar dado; aqui a recomendação é mais específica — tudo que este capítulo fez à mão (explorar, limpar, filtrar, agrupar, agregar, reescalonar) tem um método de `pandas.DataFrame` correspondente, geralmente em uma linha. *Python for Data Analysis* (O'Reilly), de Wes McKinney — o criador do `pandas` —, é a referência que o próprio autor recomenda para aprendê-lo a sério.
- O `scikit-learn` tem uma [família inteira de funções de decomposição de matriz](https://scikit-learn.org/stable/modules/decomposition.html), incluindo PCA — mas resolvida por decomposição em valores singulares, não pelo gradiente descendente que você usou na [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html). Para o tratamento formal de por que isso funciona, veja G{\'e}ron (2022).

## Referências

- **G{\'e}ron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed.. O'Reilly Media. 2022.
- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.